## תרגול

בלי להריץ, תארו במילים מה מחשב הקוד הבא, וכמה שורות תהיה לתוצאה `result`:

In [ ]:
import pandas as pd
df = pd.read_csv("launch_log.csv")

fast_runs = df[df["v0_measured"] > df["v0_measured"].mean()]
result = fast_runs.groupby("angle_deg")["range_measured"].agg(["mean", "count"])

`````{admonition} פתרון
:class: dropdown, tip
השורה הראשונה מסננת רק ריצות שבהן `v0_measured` היה **מעל הממוצע הכללי** של כל 30 הריצות (לא ממוצע לפי זווית — ממוצע אחד, גלובלי, על כל הטבלה). השורה השנייה מקבצת את הריצות שנשארו לפי `angle_deg`, ומחשבת לכל קבוצה גם ממוצע וגם מונה (`count`) של `range_measured`.

`result` היא `DataFrame` עם **עמודה אחת לכל סטטיסטיקה** (`mean`, `count`) ו**שורה אחת לכל זווית שנותרה** בנתונים המסוננים — לא בהכרח כל חמש הזוויות, כי ייתכן שבזווית מסוימת אף ריצה לא עברה את סף המהירות הממוצעת (במקרה זה, הזווית לא תופיע בתוצאה).
`````

---

הקוד הבא נועד להוסיף לטבלה עמודה חדשה `above_mean`, שמסמנת אילו ריצות היה בהן טווח מעל הממוצע. יש בו באג — אתרו ותקנו:

In [ ]:
df2 = pd.read_csv("launch_log.csv")

for i in range(len(df2)):
    if df2["range_measured"][i] > df2["range_measured"].mean():
        is_above = True
    else:
        is_above = False

df2["above_mean"] = is_above
print(df2["above_mean"].unique())

`````{admonition} פתרון
:class: dropdown, tip
המשתנה ‎`is_above`‎ מחושב **מחדש בכל איטרציה של הלולאה**, ונשמר עם הערך של השורה **האחרונה בלבד**. אחרי סיום הלולאה, ‎`df2["above_mean"] = is_above`‎ משייך לכל 30 השורות **את אותו ערך בודד** — לא ערך שונה לכל שורה. הקוד רץ בלי שגיאה, אבל התוצאה חסרת משמעות: ‎`unique()`‎ יראה רק ערך אחד (‎`True`‎ או ‎`False`‎), לא שניהם.

הפתרון הנכון: לוותר על הלולאה ולהשתמש בהשוואה וקטורית ישירה על כל העמודה בבת אחת (בדיוק כמו מסכה בוליאנית מסעיף 7.7 / 11.4):

```python
df2 = pd.read_csv("launch_log.csv")
df2["above_mean"] = df2["range_measured"] > df2["range_measured"].mean()
print(df2["above_mean"].unique())   # [True, False] - שני הערכים כצפוי
```
`````

---

כתבו פונקציה `best_angle_by_mean_range(df)` שמקבלת DataFrame בפורמט של `launch_log.csv`, ומחזירה **tuple** של (הזווית עם הטווח הממוצע הגבוה ביותר, ערך הטווח הממוצע הזה).

In [ ]:
def best_angle_by_mean_range(df):
    # כתבו כאן
    pass

angle, value = best_angle_by_mean_range(pd.read_csv("launch_log.csv"))
print(angle)              # expected: 45
print(f"{value:.1f}")     # expected: 41.6

`````{admonition} פתרון
:class: dropdown, tip
```python
def best_angle_by_mean_range(df):
    mean_by_angle = df.groupby("angle_deg")["range_measured"].mean()
    best_angle = mean_by_angle.idxmax()
    best_value = mean_by_angle.max()
    return best_angle, best_value

angle, value = best_angle_by_mean_range(pd.read_csv("launch_log.csv"))
print(angle)              # expected: 45
print(f"{value:.1f}")     # expected: 41.6
```
`````

---

### שאלה מלאה: ערכים חסרים וקיבוץ

ביומן קצר שש זריקות בשתי זוויות, ובזריקה אחת חסר הטווח (‎`np.nan`‎). **בלי להריץ**, כתבו מה תדפיס כל אחת משורות ההדפסה A עד E. את כל החישובים אפשר לעשות בראש.

In [ ]:
import numpy as np
import pandas as pd

runs = pd.DataFrame({
    "angle_deg": [30, 30, 30, 60, 60, 60],
    "v0_measured": [10.0, 20.0, 30.0, 10.0, 20.0, 30.0],
    "range_measured": [8.0, 12.0, np.nan, 9.0, 15.0, 21.0],
})
print(runs.shape)                                           # line A
fast = runs[runs["v0_measured"] > 15]
print(fast["range_measured"].mean())                        # line B
print(runs.dropna().shape)                                  # line C
print(runs.groupby("angle_deg")["range_measured"].mean())   # line D
print(runs.groupby("angle_deg")["range_measured"].count())  # line E

`````{admonition} פתרון
:class: dropdown, tip
```text
(6, 3)
16.0
(5, 3)
angle_deg
30    10.0
60    15.0
Name: range_measured, dtype: float64
angle_deg
30    2
60    3
Name: range_measured, dtype: int64
```
- **A:** שש שורות ושלוש עמודות.
- **B:** המסכה משאירה את ארבע הזריקות שבהן ‎`v0_measured`‎ הוא 20 או 30. באחת מהן הטווח חסר, ו-‎`mean()`‎ מדלג עליו: $(12+15+21)/3 = 16$.
- **C:** ‎`dropna()`‎ מסיר את השורה היחידה שיש בה ‎`NaN`‎: נשארות חמש שורות, ומספר העמודות לא משתנה.
- **D:** ‎`groupby`‎ מצמצם כל זווית לשורה אחת, ומעל הערכים מודפס שם ה-‎`index`‎, ‎`angle_deg`‎. ב-30 מעלות ‎`NaN`‎ לא נספר: $(8+12)/2 = 10$, וב-60 מעלות $(9+15+21)/3 = 15$.
- **E:** ‎`count()`‎ סופר רק ערכים שאינם חסרים: 2 ב-30 מעלות ו-3 ב-60 מעלות. הספירה היא מספר שלם, ולכן ‎`dtype: int64`‎.
`````